In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from datetime import datetime

def extraer_google_academico_misma_jerarquia(
    query="inmigracion",
    tema="INMIGRACION",
    etiqueta="VERDADERO",
    num_paginas=3,
    resultados_por_pagina=10,
    desde_anio=2025,
    archivo_salida="google_scholar_inmigracion_verdadero_tema.csv"
):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/122.0.0.0 Safari/537.36"
        )
    }

    base_url = "https://scholar.google.com/scholar"
    datos = []

    fecha_extraccion = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for pagina in range(num_paginas):
        start = pagina * resultados_por_pagina

        params = {
            "q": query,
            "hl": "es",
            "start": start,
            "as_ylo": desde_anio,
            "scisbd": 1   # ordenar por fecha
        }

        print(f"Procesando página {pagina + 1}...")

        try:
            response = requests.get(base_url, params=params, headers=headers, timeout=20)
            response.raise_for_status()
        except Exception as e:
            print(f"Error en la petición de la página {pagina + 1}: {e}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        resultados = soup.select(".gs_r.gs_or.gs_scl")

        if not resultados:
            print("No se encontraron resultados o Google Académico ha bloqueado la petición.")
            break

        for r in resultados:
            titular = ""
            cuerpo = ""
            url = ""
            fecha = ""
            fuente = "Google Académico"
            tema_articulo = tema

            # Título y enlace
            bloque_titulo = r.select_one(".gs_rt")
            if bloque_titulo:
                enlace = bloque_titulo.find("a")
                if enlace:
                    titular = enlace.get_text(" ", strip=True)
                    url = enlace.get("href", "").strip()
                else:
                    titular = bloque_titulo.get_text(" ", strip=True)

            # Resumen -> lo guardamos en Cuerpo
            bloque_resumen = r.select_one(".gs_rs")
            if bloque_resumen:
                cuerpo = bloque_resumen.get_text(" ", strip=True)

            # Meta: autores, revista, año
            bloque_meta = r.select_one(".gs_a")
            if bloque_meta:
                meta_texto = bloque_meta.get_text(" ", strip=True)

                # Intento de extraer año
                import re
                match = re.search(r"\b(19|20)\d{2}\b", meta_texto)
                if match:
                    fecha = match.group(0)

                # Usamos la parte posterior al primer guion como "Fuente" si existe
                partes = [p.strip() for p in meta_texto.split(" - ")]
                if len(partes) >= 2:
                    fuente = partes[1]

            datos.append({
                "Titular": titular,
                "Etiqueta": etiqueta,
                "Cuerpo": cuerpo,
                "Tema": tema_articulo,
                "URL": url,
                "Fecha": fecha,
                "Fuente": fuente,
                "Fecha_extraccion": fecha_extraccion
            })

        time.sleep(random.uniform(3, 6))

    df = pd.DataFrame(datos, columns=[
        "Titular",
        "Etiqueta",
        "Cuerpo",
        "Tema",
        "URL",
        "Fecha",
        "Fuente",
        "Fecha_extraccion"
    ])

    # Guardar con tabulador para mantener limpio el campo Cuerpo
    df.to_csv(archivo_salida, sep="\t", index=False, encoding="utf-8-sig")

    print(f"\nArchivo guardado correctamente: {archivo_salida}")
    print(f"Total de registros: {len(df)}")

    return df


# EJEMPLO DE USO
df_scholar = extraer_google_academico_misma_jerarquia(
    query="inmigracion",
    tema="INMIGRACION",
    etiqueta="VERDADERO",
    num_paginas=5,
    resultados_por_pagina=10,
    desde_anio=2025,
    archivo_salida="google_scholar_inmigracion_verdadero_tema.csv"
)

print(df_scholar.head())

Procesando página 1...
Procesando página 2...
Procesando página 3...
Procesando página 4...
Procesando página 5...

Archivo guardado correctamente: google_scholar_inmigracion_verdadero_tema.csv
Total de registros: 50
                                             Titular   Etiqueta  \
0            Presentación del ejemplar de la revista  VERDADERO   
1  Parte II Diversidad cultural, interculturalida...  VERDADERO   
2  Exilio, identidad y patrimonio: el legado cult...  VERDADERO   
3  La identidad de la sexta oleada de migrantes p...  VERDADERO   
4  Patrimonio simbólico y bienes patrimonializabl...  VERDADERO   

                                              Cuerpo         Tema  \
0  Hace 2 días - … Tomás Calvo Buezas, titulada *...  INMIGRACION   
1  Hace 5 días - El libro que presentamos se trat...  INMIGRACION   
2  Hace 5 días - Esta presentación se inserta en ...  INMIGRACION   
3  Hace 5 días - … En el fondo “ inmigración ” se...  INMIGRACION   
4  Hace 5 días - … Como estudiantes